## **Messwerte**

In [7]:
import numpy as np #Messwerte zu 3.1

alpha_Fehler = 0.01 #Fehler für die Messdaten der Ausnelkungswinkel [rad]

omega_Fehler = 0.01

phi_Fehler = 0.174533 #10 Grad
"""-----------------------------------------------------------------------------------------"""

#Verlauf der gedämpften Schwingung mit I = 0.075A

omega_1 = np.array([0.390,0.405, 0.436, 0.468, 0.470, 0.476, 0.489, 0.510, 0.549,0.573, 0.591, 0.593, 0.619]) *2*np.pi #Liste der Erregerfrequenzen bei denen gemessen wurde
alpha_1 = np.array([4.940, 5.6, 5.533, 8.892, 8.497, 10.47, 13.04, 47.02, 21.73, 12.05, 8.892, 8.69, 4.94]) #Auslenkungswinkel in Radians nach Einschwingung
phi_1  =  np.radians(np.array([-0.2811, 0, -0.8291, -0.119, -1.291, 8.19, 9.59, 22.01, 168.2, 168.4, 165.8, 165.8, 177.3])) #Phasenverschiebung gegebüber Erreger

"""-----------------------------------------------------------------------------------------"""

#Verlauf der gedämpften Schwingung mit I = 0.150 A

omega_2 = np.array([0.428 , 0.439 , 0.450 , 0.458 , 0.468 , 0.478 , 0.488 , 0.539 , 0.541, 0.546 , 0.552 , 0.557 , 0.580 , 0.594 , 0.619])*2*np.pi #Liste der Erregerfrequenzen bei denen gemessen wurde
alpha_2 = np.array([5.34 , 6.13 , 6.92 , 7.71 , 9.09 , 10.27 , 13.43 , 31.02 , 29.04 , 21.73 , 18.17 , 15.80 , 8.892 , 6.914 , 4.94]) #Auslenkungswinkel in Radians nach Einschwingung
phi_2  =  np.radians(np.array([4.001 , 8.59 , 4.026 , 3.301 , 7.441 , 4.362 , 14.78 , 150.9 , 159.0 , 165.3 , 164.3 , 163.4 , 169.8 , 174.1 , 177.3])) #Phasenverschiebung gegebüber Erreger #TODO[rad]

"""-----------------------------------------------------------------------------------------"""




'-----------------------------------------------------------------------------------------'

## **Dämpfung 1**

In [8]:
import matplotlib.pyplot as plt
import scipy
import sympy
from IPython.display import display
from Skripte.Fehlerfortpflanzung import Gaußfehler
import matplotlib


matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'arial',
    'text.usetex': True,
    'pgf.rcfonts': False,
})


fig, ax1 = plt.subplots(1,1, figsize = (4,4))

ax1.errorbar(omega_1, alpha_1, yerr = alpha_Fehler, fmt = "o", label = "Datensatz", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")


fit = scipy.interpolate.Akima1DInterpolator(omega_1, alpha_1)

xwerte = np.linspace(np.min(omega_1), np.max(omega_1), 500)
ax1.plot(xwerte, fit.__call__(xwerte), color = "blue", linestyle = "--", label = "Interpolation")

ax1.set_xlabel("$\\omega~[1/s]$")
ax1.set_ylabel("$A_1~[\\mathrm{V}]$")
ax1.grid(True)
ax1.set_title("Amplitude - Anregerfrequenz(1)")


"""----------------------------------------------------------------"""

omega_res_1 = fit.solve(np.max(fit.__call__(xwerte)))[0] #Resonanzfrequenz
#ax1.vlines(omega_res_1, ymin = 0, ymax = 55, color = "grey")
omega_res_1_Fehler = 0.1 #wegen wenig möglichen Messungen um die Resonanzfrequenz

print(f"Resonanzfrequenz: omega_Res = {omega_res_1} +/- {omega_res_1_Fehler}\n")

A_1 = fit.__call__(omega_res_1) #maximum Amplitude
#ax1.axhline((1/np.sqrt(2))*A_1, label = "Halbwertsbreite")
ax1.legend()

points_1 = fit.solve((1/np.sqrt(2))*A_1)
#ax1.vlines(points_1[0], ymin = 0, ymax = 55, color = "red")
#ax1.vlines(points_1[1], ymin = 0, ymax = 55, color = "red")
delta_omega_1 = points_1[1] - points_1[0]
delta_omega_1_Fehler = 0.1

print(f"Halbwertsbreite (abgelesen): deltaOmega = {delta_omega_1} +/- {delta_omega_1_Fehler}\n")

lmbda_1 = 0.5 * delta_omega_1
lmbda_1_Fehler = 0.5 * delta_omega_1_Fehler

print(f"Dämpfungskonstante: lmbda = {lmbda_1} +/- {lmbda_1_Fehler}\n")

omega_Eigen_1 = np.sqrt(omega_res_1**2 + 2*(lmbda_1**2))

wr, l1 = sympy.symbols("omega_Res, lambda")
expr1 = sympy.sqrt(wr**2 + 2*(l1**2))
display(expr1)

omega_Eigen_1_Fehler = Gaußfehler(expr1, np.array([wr, l1]), np.array([omega_res_1, lmbda_1]), np.array([omega_res_1_Fehler, lmbda_1_Fehler]))

print(f"Eigenfrequenz: omega_Eigen = {omega_Eigen_1} +/- {omega_Eigen_1_Fehler} 1/s \n")
plt.savefig("Erzwungen11.pgf")

"""-------------------------------------------------------------------------------------------------------------"""
fig, ax2 = plt.subplots(1,1, figsize = (4,4))

ax2.errorbar(omega_1, phi_1, yerr = phi_Fehler, fmt = "o", label = "Datensatz", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")

#ax2.axhline(np.pi/2, label = "Eigenfrequenz") #Hier Anregung mit Eigenfrequenz


fit = scipy.interpolate.Akima1DInterpolator(omega_1, phi_1)
fit_deriv = fit.derivative()

xwerte = np.linspace(np.min(omega_1), np.max(omega_1), 500)
ax2.plot(xwerte, fit.__call__(xwerte), color = "blue", linestyle = "--", label = "Interpolation")

omega_Eigen_phi_1 = fit.solve(np.pi/2)[0] #Eigenfrequenz aus dem Phi Diagramm

print(f"Eigenfrequenz aus der Phasenverschiebung: omega_Eigen_phi = {omega_Eigen_phi_1}\n")

m_1 = fit_deriv.__call__(omega_Eigen_phi_1) #Steigung bei Eigenfrequenz im Phasenverschiebungsdiagramm

lmbda_phi_1 = 1/m_1 #Lambda aus der Steigung im Phasendiagramm

print(f"Dämpfungskonstante aus Phasenverschiebung: lmbda = {lmbda_phi_1} +/- 0.2617 \n") #Fehler: 15 Grad

omega_res_phi_1 = np.sqrt(omega_Eigen_phi_1**2 - 2*(lmbda_phi_1**2))


wep, lmbp = sympy.symbols("omega_eig, lambda_phi")
exprT = sympy.sqrt(wep**2 - 2* (lmbp**2))
omega_res_phi_1_Fehler = Gaußfehler(exprT, np.array([wep, lmbp]), np.array([omega_Eigen_phi_1, lmbda_phi_1]), np.array([omega_Eigen_1_Fehler, 0.2617]))

print(f"Resonanzfrequenz aus Phasenverschienung: omega_Res = {omega_res_phi_1} +/- {omega_res_phi_1_Fehler}  \n")



ax2.set_xlabel("$\\omega~[1/s]$")
ax2.set_ylabel("$\\varphi~[rad]$")
ax2.grid(True)
ax2.set_title("Phasenverschiebung - Anregerfrequenz(1)")
ax2.legend()
plt.savefig("Erzwungen12.pgf")


Resonanzfrequenz: omega_Res = 3.200143378396173 +/- 0.1

Halbwertsbreite (abgelesen): deltaOmega = 0.1966873013181698 +/- 0.1

Dämpfungskonstante: lmbda = 0.0983436506590849 +/- 0.05



sqrt(2*lambda**2 + omega_Res**2)

Eigenfrequenz: omega_Eigen = 3.203164152762512 +/- 0.0999528581475397 1/s 

Eigenfrequenz aus der Phasenverschiebung: omega_Eigen_phi = 3.3165196562224666

Dämpfungskonstante aus Phasenverschiebung: lmbda = 0.06744743757502164 +/- 0.2617 

Resonanzfrequenz aus Phasenverschienung: omega_Res = 3.3151477065794692 +/- 0.10055963041272829  



## **Dämpfung 2**

In [9]:

fig, ax1 = plt.subplots(1,1, figsize = (4,4))


matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'arial',
    'text.usetex': True,
    'pgf.rcfonts': False,
})

ax1.errorbar(omega_2, alpha_2, yerr = alpha_Fehler, fmt = "o", label = "Datensatz", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")


fit = scipy.interpolate.CubicSpline(omega_2, alpha_2) #TODO: Cubic Spline weil keine Mesung um w0 möglich - Höherer Fehler für lambda

xwerte = np.linspace(np.min(omega_2), np.max(omega_2), 500)
ax1.plot(xwerte, fit.__call__(xwerte), color = "blue", linestyle = "--", label = "Interpolation")

ax1.set_xlabel("$\\omega~[1/s]$")
ax1.set_ylabel("$A_1~[\\mathrm{V}]$")
ax1.grid(True)
ax1.set_title("Amplitude - Anregerfrequenz(2)")


"""----------------------------------------------------------------"""

omega_res_2 = fit.solve(np.max(fit.__call__(xwerte)))[0] #Resonanzfrequenz
#ax1.vlines(omega_res_2, ymin = 0, ymax = 55, color = "grey")
omega_res_2_Fehler = 0.1 #wegen wenig möglichen Messungen um die Resonanzfrequenz

print(f"Resonanzfrequenz: omega_Res = {omega_res_2} +/- {omega_res_2_Fehler}\n")

A_2 = fit.__call__(omega_res_2) #maximum Amplitude
#ax1.axhline((1/np.sqrt(2))*A_2, label = "Halbwertsbreite")
ax1.legend()

points_2 = fit.solve((1/np.sqrt(2))*A_2)
#ax1.vlines(points_2[0], ymin = 0, ymax = 55, color = "red")
#ax1.vlines(points_2[1], ymin = 0, ymax = 55, color = "red")
delta_omega_2 = points_2[1] - points_2[0]
delta_omega_2_Fehler = 0.1

print(f"Halbwertsbreite (abgelesen): deltaOmega = {delta_omega_2} +/- {delta_omega_2_Fehler}\n")

lmbda_2 = 0.5 * delta_omega_2
lmbda_2_Fehler = 0.5 * delta_omega_2_Fehler

print(f"Dämpfungskonstante: lmbda = {lmbda_2} +/- {lmbda_2_Fehler}\n")

omega_Eigen_2 = np.sqrt(omega_res_2**2 + 2*(lmbda_2**2))

wr, l1 = sympy.symbols("omega_Res, lambda")
expr1 = sympy.sqrt(wr**2 + 2*(l1**2))
display(expr1)

omega_Eigen_2_Fehler = Gaußfehler(expr1, np.array([wr, l1]), np.array([omega_res_2, lmbda_2]), np.array([omega_res_2_Fehler, lmbda_2_Fehler]))

print(f"Eigenfrequenz: omega_Eigen = {omega_Eigen_2} +/- {omega_Eigen_2_Fehler} 1/s \n")

plt.savefig("Erzwungen21.pgf")

"""-------------------------------------------------------------------------------------------------------------"""

fig, ax2 = plt.subplots(1,1, figsize = (4,4))

ax2.errorbar(omega_2, phi_2, yerr = phi_Fehler, fmt = "o", label = "Datensatz", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")

#ax2.axhline(np.pi/2, label = "Eigenfrequenz") #Hier Anregung mit Eigenfrequenz


fit = scipy.interpolate.Akima1DInterpolator(omega_2, phi_2)
fit_deriv = fit.derivative()

xwerte = np.linspace(np.min(omega_2), np.max(omega_2), 500)
ax2.plot(xwerte, fit.__call__(xwerte), color = "blue", linestyle = "--", label = "Interpolation")

omega_Eigen_phi_2 = fit.solve(np.pi/2)[0] #Eigenfrequenz aus dem Phi Diagramm

print(f"Eigenfrequenz aus der Phasenverschiebung: omega_Eigen_phi = {omega_Eigen_phi_2}\n")

m_2 = fit_deriv.__call__(omega_Eigen_phi_2) #Steigung bei Eigenfrequenz im Phasenverschiebungsdiagramm

lmbda_phi_2 = 1/m_2 #Lambda aus der Steigung im Phasendiagramm

print(f"Dämpfungskonstante aus Phasenverschiebung: lmbda = {lmbda_phi_2} +/- 0.2617 \n") #Fehler 15 Grad

omega_res_phi_2 = np.sqrt(omega_Eigen_phi_2**2 - 2*(lmbda_phi_2**2))

wep, lmbp = sympy.symbols("omega_eig, lambda_phi")
exprT = sympy.sqrt(wep**2 - 2* (lmbp**2))
omega_res_phi_2_Fehler = Gaußfehler(exprT, np.array([wep, lmbp]), np.array([omega_Eigen_phi_2, lmbda_phi_2]), np.array([omega_Eigen_2_Fehler, 0.2617]))

print(f"Resonanzfrequenz aus Phasenverschienung: omega_Res = {omega_res_phi_2} +/- {omega_res_phi_2_Fehler} \n")

ax2.set_xlabel("$\\omega~[1/s]$")
ax2.set_ylabel("$\\phi~[rad]$")
ax2.grid(True)
ax2.set_title("Phasenverschiebung - Anregerfrequenz(2)")
ax2.legend()
plt.savefig("Erzwungen22.pgf")

Resonanzfrequenz: omega_Res = 3.3144998692975864 +/- 0.1

Halbwertsbreite (abgelesen): deltaOmega = 0.24029049451035034 +/- 0.1

Dämpfungskonstante: lmbda = 0.12014524725517517 +/- 0.05



sqrt(2*lambda**2 + omega_Res**2)

Eigenfrequenz: omega_Eigen = 3.318852082339575 +/- 0.09993445346210864 1/s 

Eigenfrequenz aus der Phasenverschiebung: omega_Eigen_phi = 3.2613283439222545

Dämpfungskonstante aus Phasenverschiebung: lmbda = 0.12493635264070682 +/- 0.2617 

Resonanzfrequenz aus Phasenverschienung: omega_Res = 3.256538711952976 +/- 0.1020759752593827 



C:\Users\Felix\AppData\Local\Temp\ipykernel_9572\780514323.py:102: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  fig.show()
